# 04 — Geo + Graph features

Features that live *between* localities, not within one row.

- **Spatial:** distance to city centroid, k-NN density (BallTree, haversine), DBSCAN spatial clusters
- **Adjacency graph** from the "Nearby Localities" column (same-city name matching):
  degree, PageRank, clustering coefficient, and **Louvain communities = contiguous "belts"**
  (directly serves "attack this belt").

*node2vec structural embeddings are intentionally skipped:* its `gensim` dependency has no Python 3.13
wheel here. The centrality + belt features below carry the actionable structural signal; node2vec can be
added later in an environment with a gensim build.

Reads `artifacts/features_text.parquet`; writes `artifacts/features_geo_graph.parquet`.

In [1]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

ART = Path.cwd() / "artifacts"
df = pd.read_parquet(ART / "features_text.parquet").reset_index(drop=True)
print("Loaded:", df.shape)
print("Geocoded:", int(df["lat"].notna().sum()), "/", len(df))

Loaded: (1001, 61)
Geocoded: 886 / 1001


In [2]:
# --- Spatial features ---
from sklearn.neighbors import BallTree
from sklearn.cluster import DBSCAN

EARTH_KM = 6371.0

def haversine_km(lat1, lng1, lat2, lng2):
    lat1, lng1, lat2, lng2 = map(np.radians, [lat1, lng1, lat2, lng2])
    dlat, dlng = lat2 - lat1, lng2 - lng1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlng / 2) ** 2
    return EARTH_KM * 2 * np.arcsin(np.sqrt(a))

# distance to the centroid of the locality's own city
df["dist_to_city_centroid_km"] = np.nan
for city, grp in df.groupby("ADDRESS"):
    geo = grp.dropna(subset=["lat", "lng"])
    if geo.empty:
        continue
    clat, clng = geo["lat"].mean(), geo["lng"].mean()
    df.loc[geo.index, "dist_to_city_centroid_km"] = haversine_km(geo["lat"].values, geo["lng"].values, clat, clng).round(2)

# k-NN density: mean great-circle distance to the 5 nearest localities (smaller = denser)
geo = df.dropna(subset=["lat", "lng"])
coords = np.radians(geo[["lat", "lng"]].values)
tree = BallTree(coords, metric="haversine")
k = min(6, len(coords))
dist, _ = tree.query(coords, k=k)  # first neighbour is self (0)
df["knn_density_km"] = np.nan
df.loc[geo.index, "knn_density_km"] = (dist[:, 1:].mean(axis=1) * EARTH_KM).round(2)

# DBSCAN spatial clusters (~2 km radius)
dbscan = DBSCAN(eps=2.0 / EARTH_KM, min_samples=3, metric="haversine")
df["spatial_cluster"] = -1
df.loc[geo.index, "spatial_cluster"] = dbscan.fit_predict(coords)
print("Spatial clusters:", df.loc[df.spatial_cluster >= 0, "spatial_cluster"].nunique(),
      "| noise:", int((df.spatial_cluster == -1).sum()))
print(df[["dist_to_city_centroid_km", "knn_density_km"]].describe().round(2).to_string())

Spatial clusters: 79 | noise: 271
       dist_to_city_centroid_km  knn_density_km
count                    886.00          886.00
mean                      15.62            2.53
std                       19.81           10.89
min                        0.32            0.00
25%                        5.50            0.00
50%                        9.45            0.00
75%                       20.24            2.05
max                      298.27          275.05


In [3]:
# --- Build the locality adjacency graph from "Nearby Localities" (same-city name matching) ---
import networkx as nx

df["area_core"] = df["AREA"].astype(str).str.split(",").str[0].str.strip().str.lower()

G = nx.Graph()
G.add_nodes_from(df.index)
for city, grp in df.groupby("ADDRESS"):
    cores = {i: c for i, c in grp["area_core"].items() if isinstance(c, str) and len(c) > 3}
    texts = {i: (str(t).lower() if pd.notna(t) else "") for i, t in grp["Nearby Localities"].items()}
    for i, ti in texts.items():
        if not ti:
            continue
        for j, cj in cores.items():
            if i != j and cj in ti:
                G.add_edge(i, j)

print("Graph:", G.number_of_nodes(), "nodes,", G.number_of_edges(), "edges")
print("Connected components:", nx.number_connected_components(G))

Graph: 1001 nodes, 965 edges
Connected components: 384


In [4]:
# --- Graph features + Louvain belts ---
import community as community_louvain

deg = dict(G.degree())
pr = nx.pagerank(G) if G.number_of_edges() else {n: 0.0 for n in G.nodes()}
clus = nx.clustering(G)
partition = community_louvain.best_partition(G, random_state=42) if G.number_of_edges() else {n: n for n in G.nodes()}

df["graph_degree"] = df.index.map(deg).astype(int)
df["graph_pagerank"] = df.index.map(pr).round(6)
df["graph_clustering"] = df.index.map(clus).round(4)
df["belt_id"] = df.index.map(partition).astype(int)
belt_sizes = df["belt_id"].value_counts()
df["belt_size"] = df["belt_id"].map(belt_sizes).astype(int)

# only multi-node belts are meaningful "belts"
multi = belt_sizes[belt_sizes > 1]
print("Belts (>=2 localities):", len(multi), "| largest belt:", int(belt_sizes.max()))
top_belt = belt_sizes.index[0]
print("Example largest belt members:",
      ", ".join(df.loc[df.belt_id == top_belt, "AREA"].head(6).tolist()))

Belts (>=2 localities): 114 | largest belt: 60
Example largest belt members: Ardee City, Gurgaon, Block A Sector 47, Gurgaon, Block A Sushant Lok Phase 1, Gurgaon, Block B Sushant Lok Phase 1, Gurgaon, Block C Sushant Lok Phase 1, Gurgaon, Golf Course Extension Road, Gurgaon


In [5]:
# --- Report + save ---
new = ["dist_to_city_centroid_km", "knn_density_km", "spatial_cluster",
       "graph_degree", "graph_pagerank", "graph_clustering", "belt_id", "belt_size"]
cov = pd.DataFrame({"non_null": [df[c].notna().sum() for c in new]}, index=new)
print(cov.to_string())

out = ART / "features_geo_graph.parquet"
df.to_parquet(out, index=False)
print("\nSaved", df.shape[0], "rows x", df.shape[1], "cols ->", out.relative_to(Path.cwd()))

                          non_null
dist_to_city_centroid_km       886
knn_density_km                 886
spatial_cluster               1001
graph_degree                  1001
graph_pagerank                1001
graph_clustering              1001
belt_id                       1001
belt_size                     1001

Saved 1001 rows x 70 cols -> artifacts\features_geo_graph.parquet


## Output
`artifacts/features_geo_graph.parquet` — spatial density/centroid distance/clusters + graph centrality,
PageRank, clustering, and Louvain **belts** (contiguous locality groups to attack together).

**Next:** `05_unsupervised_segmentation.ipynb` — cluster the full feature + embedding matrix (KMeans / HDBSCAN
+ PCA/UMAP) to derive data-driven locality archetypes.